In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import optuna
from sklearn.metrics import mean_squared_log_error

DATA_DIR = "../data"

train = pd.read_parquet(f"{DATA_DIR}/train_processed.parquet")
train = train.sort_values("date")

categorical_cols = ["store_nbr", "family", "city", "state", "type", "cluster"]
for col in categorical_cols:
    train[col] = train[col].astype("category")

exclude_cols = ["id", "date", "sales", "sales_log", "is_train"]
feature_cols = [col for col in train.columns if col not in exclude_cols]

print("Features:", len(feature_cols))
print("Train shape:", train.shape)

Features: 22
Train shape: (3000888, 27)


In [2]:
from sklearn.model_selection import TimeSeriesSplit

tscv = TimeSeriesSplit(n_splits=3, test_size=16*1782)  # 16 dni * liczba kombinacji store+family

for fold, (train_idx, val_idx) in enumerate(tscv.split(train)):
    fold_train = train.iloc[train_idx]
    fold_val = train.iloc[val_idx]
    print(f"Fold {fold}: train dates {fold_train['date'].min()} - {fold_train['date'].max()}, "
          f"val dates {fold_val['date'].min()} - {fold_val['date'].max()}")

Fold 0: train dates 2013-01-01 00:00:00 - 2017-06-28 00:00:00, val dates 2017-06-29 00:00:00 - 2017-07-14 00:00:00
Fold 1: train dates 2013-01-01 00:00:00 - 2017-07-14 00:00:00, val dates 2017-07-15 00:00:00 - 2017-07-30 00:00:00
Fold 2: train dates 2013-01-01 00:00:00 - 2017-07-30 00:00:00, val dates 2017-07-31 00:00:00 - 2017-08-15 00:00:00


In [3]:
def rmsle_from_log(y_true_log, y_pred_log):
    y_true = np.expm1(y_true_log)
    y_pred = np.clip(np.expm1(y_pred_log), 0, None)
    return np.sqrt(mean_squared_log_error(y_true, y_pred))


scores = []

for fold, (train_idx, val_idx) in enumerate(tscv.split(train)):
    fold_train = train.iloc[train_idx]
    fold_val = train.iloc[val_idx]

    X_train = fold_train[feature_cols]
    y_train = fold_train["sales_log"]
    X_val = fold_val[feature_cols]
    y_val = fold_val["sales_log"]

    model = lgb.LGBMRegressor(
        n_estimators=500,
        learning_rate=0.05,
        num_leaves=31,
        random_state=42,
        verbosity=-1
    )

    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        eval_metric="rmse",
        callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
    )

    val_pred_log = model.predict(X_val)
    score = rmsle_from_log(y_val, val_pred_log)
    scores.append(score)
    print(f"Fold {fold} RMSLE: {score:.4f}")

print(f"\nMean RMSLE across folds: {np.mean(scores):.4f} (+/- {np.std(scores):.4f})")

C:\Users\hyuhi\projects\store-sales-forecasting\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Fold 0 RMSLE: 0.3734


C:\Users\hyuhi\projects\store-sales-forecasting\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Fold 1 RMSLE: 0.3800


C:\Users\hyuhi\projects\store-sales-forecasting\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Fold 2 RMSLE: 0.3854

Mean RMSLE across folds: 0.3796 (+/- 0.0049)


In [4]:
model.fit(
    X_train, y_train,
    eval_X=X_val, eval_y=y_val,
    eval_metric="rmse",
    callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
)

,learning_rate,0.05
,n_estimators,500
,random_state,42
,verbosity,-1
,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0


In [5]:
train_idx_list = list(tscv.split(train))
train_idx, val_idx = train_idx_list[-1]  # last fold, closest to real test period

fold_train = train.iloc[train_idx]
fold_val = train.iloc[val_idx]

X_train = fold_train[feature_cols]
y_train = fold_train["sales_log"]
X_val = fold_val[feature_cols]
y_val = fold_val["sales_log"]


def objective(trial):
    params = {
        "n_estimators": 1000,
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 20, 150),
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
        "random_state": 42,
        "verbosity": -1
    }

    model = lgb.LGBMRegressor(**params)
    model.fit(
        X_train, y_train,
        eval_X=X_val, eval_y=y_val,
        eval_metric="rmse",
        callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
    )

    val_pred_log = model.predict(X_val)
    score = rmsle_from_log(y_val, val_pred_log)
    return score


study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=30, show_progress_bar=True)

print("Best RMSLE:", study.best_value)
print("Best params:", study.best_params)

[I 2026-09-24 23:03:40,644] A new study created in memory with name: no-name-00a86ea8-ea99-402c-9e96-eb256af78238


  0%|          | 0/30 [00:00<?, ?it/s]

[I 2026-09-24 23:04:46,514] Trial 0 finished with value: 0.38379488358833136 and parameters: {'learning_rate': 0.01891773117024482, 'num_leaves': 123, 'max_depth': 6, 'min_child_samples': 17, 'subsample': 0.828726789809929, 'colsample_bytree': 0.7109149694418861, 'reg_alpha': 0.001647459315514849, 'reg_lambda': 0.14363857682028724}. Best is trial 0 with value: 0.38379488358833136.
[I 2026-09-24 23:05:18,973] Trial 1 finished with value: 0.38073515332812496 and parameters: {'learning_rate': 0.0783797714601054, 'num_leaves': 125, 'max_depth': 11, 'min_child_samples': 17, 'subsample': 0.7518591191959438, 'colsample_bytree': 0.9094857010965782, 'reg_alpha': 5.266389564552208, 'reg_lambda': 0.18589685626009908}. Best is trial 1 with value: 0.38073515332812496.
[I 2026-09-24 23:06:29,708] Trial 2 finished with value: 0.37972098316411007 and parameters: {'learning_rate': 0.027862135721549497, 'num_leaves': 122, 'max_depth': 11, 'min_child_samples': 59, 'subsample': 0.8688610336055744, 'colsam

In [6]:
print("Best RMSLE:", study.best_value)
print("Best params:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")

Best RMSLE: 0.3755166105281336
Best params:
  learning_rate: 0.04039161784455703
  num_leaves: 131
  max_depth: 10
  min_child_samples: 43
  subsample: 0.8245394707857479
  colsample_bytree: 0.9222457859658849
  reg_alpha: 0.01773933884749932
  reg_lambda: 2.2852374507917768


In [7]:
best_params = study.best_params
best_params["random_state"] = 42
best_params["verbosity"] = -1

scores_tuned = []

for fold, (train_idx, val_idx) in enumerate(tscv.split(train)):
    fold_train = train.iloc[train_idx]
    fold_val = train.iloc[val_idx]

    X_train = fold_train[feature_cols]
    y_train = fold_train["sales_log"]
    X_val = fold_val[feature_cols]
    y_val = fold_val["sales_log"]

    model = lgb.LGBMRegressor(n_estimators=1000, **best_params)
    model.fit(
        X_train, y_train,
        eval_X=X_val, eval_y=y_val,
        eval_metric="rmse",
        callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
    )

    val_pred_log = model.predict(X_val)
    score = rmsle_from_log(y_val, val_pred_log)
    scores_tuned.append(score)
    print(f"Fold {fold} RMSLE (tuned): {score:.4f}")

print(f"\nMean RMSLE (tuned): {np.mean(scores_tuned):.4f} (+/- {np.std(scores_tuned):.4f})")
print(f"Mean RMSLE (default params): 0.3796 (+/- 0.0049)")

Fold 0 RMSLE (tuned): 0.3661
Fold 1 RMSLE (tuned): 0.3733
Fold 2 RMSLE (tuned): 0.3755

Mean RMSLE (tuned): 0.3716 (+/- 0.0040)
Mean RMSLE (default params): 0.3796 (+/- 0.0049)
